# S₂ transition locator — central 4-qubit plaquette (3D toric code)

**Independent** pseudo-critical estimate to cross-check the Fredenhagen–Marcu pipeline
(`fm_crossing_R1.ipynb`, not yet converged). We measure the second Rényi entropy
`S₂(A) = −ln Tr(ρ_A²)` of a small fixed central patch (following arXiv:2405.17541,
4-qubit central square in 2D); its derivative peak vs `h_z` gives `h_c(L)`, extrapolated
as `h_c(L) = h_c + a·L^(−x)`.

⚠️ **S₂ of a 4-qubit patch is a LOCAL, susceptibility-like transition locator — NOT a
TEE / topological / long-range-entanglement diagnostic.** The patch is far too small to
resolve a constant topological term; we use only the *location* of its derivative peak.

Patch = one central unit plaquette (4 coplanar edges = a single `B_p`), averaged over the
xy/xz/yz orientations. Exact limits: `S₂ = 3 ln2` at `h_z=0` (stabilizer GS), `0` at
`h_z→∞` (product state).

Sections: **0** geometry/limits unit test (run first, in the repo venv) · **1** load ·
**2** derivative two ways (spline + finite-difference) · **3** bootstrap peak errors ·
**4** extrapolation + robustness battery · **5** QMC comparison · **6** cross-pipeline
figure (S₂ vs FM, both vs 1/L).

In [ ]:
%matplotlib inline
import glob, json, os
import numpy as np
import matplotlib.pyplot as plt
try:
    from scipy.interpolate import UnivariateSpline
    from scipy.optimize import curve_fit
    HAVE_SCIPY = True
except Exception as _e:
    HAVE_SCIPY = False
    print("scipy unavailable -> spline derivative + weighted fits disabled:", _e)

## Config — the one cell to edit

In [ ]:
# ---- data ----
ROOT     = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main"
DATA_DIR = f"{ROOT}/results/phase_hx0.0_s2plaq"     # S2 curves (nersc/extract_s2.sh, TAG s2plaq)
FM_DIR   = f"{ROOT}/results/phase_hx0.0_bulkA0.5"   # FM aspect-1/2 curves, for the cross-pipeline plot
LS       = [4, 5, 6, 7]

# ---- derivative + peak ----
SMOOTH   = 3.0e-3        # UnivariateSpline smoothing s, FIXED across L (explicit, not per-L auto)
DENSE    = 400           # dense grid pts for locating the derivative peak
B_BOOT   = 2000          # bootstrap resamples for the peak-position error

# ---- extrapolation h_c(L) = h_c + a * L**(-x) ----
X_VARIANTS = [1.0, 1.59, 2.0]   # fixed-x robustness (1/nu approx 1.59 for the 3D Ising* class)

# ---- reference ----
QMC_HC = 0.197
QMC_NORM_NOTE = (
    "QMC h_c=0.197 at hx=0. Normalization convention (CLAUDE.md): "
    "H = -J*sum A_v - J*sum B_p - hx*sum sigma^x - hz*sum sigma^z, with J=1. "
    "Numeric value inherited from the FM notebook (HZ_C_REF) -- SOURCE TO CONFIRM."
)

## 0 · Geometry + exactly-solvable limits (run FIRST, in the repo venv)

Local unit test — **no ED**. `verify_s2_geometry` (GF(2) linear algebra on the stabilizer
generators) checks the central-plaquette patch is interior and gives the exact
stabilizer-state entropy `S₂ = 3 ln2` at `h_z=0`; the trivial product state gives `0` at
`h_z→∞`. Then, if the NQS curves are loaded, we compare the measured endpoints
`S₂(h_z^min)`, `S₂(h_z^max)` to `(3 ln2, 0)`.

*Estimator-variance note:* the SWAP estimator's relative error is amplified by `exp(S₂)`,
but a 4-qubit patch has `S₂ ≤ 4 ln2 ≈ 2.77`, so this is benign — no variance reduction
needed. Run this notebook with the repo venv kernel so `Three_TC` imports (the FM notebook's
§0 only *skipped* because it ran outside the venv — do not repeat that).

In [ ]:
try:
    from Three_TC.model.geometry import ThreeD_ToricCodeGeometry as _Geo
    from Three_TC.renyi import verify_s2_geometry, S2_EXACT_HZ0, S2_EXACT_HZINF
    print(f"exact limits:  S2(hz=0) = 3 ln2 = {S2_EXACT_HZ0:.4f}    S2(hz->inf) = {S2_EXACT_HZINF:.1f}\n")
    print(f"{'L':>3} {'xy edges':>18} {'N_A':>3} {'interior':>8} {'rankG':>6} {'rankGB':>6} {'S2_exact':>9}")
    ok_all = True
    for L in LS:
        g = _Geo(L, L, L, "OBC"); r = verify_s2_geometry(g); ok_all &= r["ok"]
        xy = r["per_plane"]["xy"]
        print(f"{L:>3} {str(xy['edges']):>18} {xy['N_A']:>3} {str(xy['interior']):>8} "
              f"{xy['rank_G']:>6} {xy['rank_GB']:>6} {xy['S2_nats']:>9.4f}"
              + ("" if r["ok"] else "   <-- FAIL"))
    print("\nGEOMETRY UNIT TEST:", "PASS" if ok_all else "FAIL  <-- patch/limit broken")
except Exception as e:
    print("Three_TC unavailable -> run this notebook with the repo venv kernel "
          "(.venv/bin/python -m ipykernel), or from the shell:")
    print("  .venv/bin/python -c \"from Three_TC.model.geometry import ThreeD_ToricCodeGeometry as G;"
          " from Three_TC.renyi import verify_s2_geometry as v;"
          " [print(L, v(G(L,L,L,'OBC'))['ok']) for L in (4,5,6,7)]\"")
    print(f"  (import error: {type(e).__name__}: {e})")

## 1 · Load S₂(h_z) for L=4,5,6,7 on a common grid

In [ ]:
def load_curves(directory, sizes=None):
    recs = [json.load(open(jp)) for jp in sorted(glob.glob(os.path.join(directory, "s2_L*.json")))]
    if not recs:
        raise SystemExit(f"no s2_L*.json in {directory} -- extract first:\n"
                         f"  PLANES=xy,xz,yz HX=0.0 LS=\"4\" bash nersc/extract_s2.sh   (one job per L)\n"
                         f"  then pull into {directory}")
    recs = sorted(recs, key=lambda r: r["L"])
    if sizes is not None:
        recs = [r for r in recs if r["L"] in sizes]
    return recs

def get_arrays(rec):
    hz = np.array(rec["field"], float)
    y  = np.array(rec["S2"], float)
    ye = np.array(rec.get("S2e", np.zeros_like(y)), float)
    order = np.argsort(hz)
    return hz[order], y[order], ye[order]

try:
    recs = load_curves(DATA_DIR, sizes=set(LS))
    curves = {r["L"]: get_arrays(r) for r in recs}
    rec_by_L = {r["L"]: r for r in recs}
    print(f"loaded L = {sorted(curves)}   from {os.path.basename(DATA_DIR)}")
    for L, (hz, y, ye) in sorted(curves.items()):
        fb = "  [sampler_fallback]" if rec_by_L[L].get("sampler_fallback") else ""
        bad = "" if np.all(np.isfinite(ye) & (ye > 0)) else "   [WARN nan/zero errbars]"
        print(f"  L={L}: {len(hz):2d} pts  hz in [{hz.min():.3g},{hz.max():.3g}]  "
              f"S2 in [{y.min():.3f},{y.max():.3f}]{fb}{bad}")
    LS_HAVE = sorted(curves)
    common = np.array(sorted(set(np.round(curves[LS_HAVE[0]][0], 6)).intersection(
        *[set(np.round(curves[L][0], 6)) for L in LS_HAVE[1:]]))) if len(LS_HAVE) > 1 \
        else np.round(curves[LS_HAVE[0]][0], 6)
    print(f"\ncommon hz grid ({len(common)} pts): [{common.min():.3g}, {common.max():.3g}]")
except SystemExit as e:
    curves = {}; LS_HAVE = []; common = np.array([])
    print(e)

## 2 · Derivative two ways → per-L peak `h_c(L)`

**(a)** smoothing cubic spline (`UnivariateSpline`, fixed `s=SMOOTH`, weighted by `1/S2e`)
differentiated on a dense grid; **(b)** centered finite differences on the raw grid. The
peak of `|dS₂/dh_z|` locates `h_c(L)` (S₂ *falls* across the transition, so the derivative
dips). If the two methods disagree by **more than the grid step**, that flags
under-resolution (densify the grid), not physics.

In [ ]:
def deriv_peak_spline(hz, y, ye, s=SMOOTH, dense=DENSE):
    if not HAVE_SCIPY:
        return np.nan, None
    w = 1.0 / np.where(np.isfinite(ye) & (ye > 0), ye, np.nanmedian(ye[ye > 0]) if np.any(ye > 0) else 1.0)
    spl = UnivariateSpline(hz, y, w=w, k=3, s=s)
    xx = np.linspace(hz.min(), hz.max(), dense)
    d = spl.derivative()(xx)
    return float(xx[np.argmax(np.abs(d))]), (xx, spl(xx), d)

def deriv_peak_fd(hz, y):
    hm = 0.5 * (hz[1:] + hz[:-1]); d = np.diff(y) / np.diff(hz)
    return float(hm[np.argmax(np.abs(d))]), (hm, d)

hc_spline, hc_fd, curve_cache = {}, {}, {}
if curves:
    grid_step = float(np.median(np.diff(common))) if len(common) > 1 else np.nan
    print(f"grid step Delta = {grid_step:.4f}   SMOOTH(s) = {SMOOTH}\n")
    print(f"{'L':>3} {'h_c spline':>11} {'h_c FD':>8} {'|diff|':>7} {'flag':>6}")
    for L in LS_HAVE:
        hz, y, ye = curves[L]
        hcs, cs = deriv_peak_spline(hz, y, ye); hcf, cf = deriv_peak_fd(hz, y)
        hc_spline[L], hc_fd[L], curve_cache[L] = hcs, hcf, (cs, cf)
        diff = abs(hcs - hcf) if np.isfinite(hcs) else np.nan
        flag = "" if (np.isfinite(diff) and diff <= grid_step) else "UNDER-RES"
        print(f"{L:>3} {hcs:>11.4f} {hcf:>8.4f} {diff:>7.4f} {flag:>6}")

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    colors = {L: c for L, c in zip(LS_HAVE, plt.cm.viridis(np.linspace(0, 0.85, len(LS_HAVE))))}
    for L in LS_HAVE:
        hz, y, ye = curves[L]; cs, cf = curve_cache[L]
        ax[0].errorbar(hz, y, yerr=ye, fmt="o", ms=4, capsize=2, color=colors[L], label=f"L={L}")
        if cs is not None:
            ax[0].plot(cs[0], cs[1], "-", color=colors[L], lw=1.4)
            ax[1].plot(cs[0], cs[2], "-", color=colors[L], lw=1.6, label=f"L={L} spline")
        ax[1].plot(cf[0], cf[1], ":", color=colors[L], lw=1, alpha=0.8)
        if np.isfinite(hc_spline[L]):
            ax[1].axvline(hc_spline[L], ls="--", lw=0.9, color=colors[L])
    ax[0].axhline(3*np.log(2), ls=":", color="gray", lw=1); ax[0].axhline(0, ls=":", color="gray", lw=1)
    for a in ax:
        a.axvline(QMC_HC, ls="--", color="green", lw=1.2, label=f"QMC={QMC_HC}")
    ax[0].set(xlabel="$h_z$", ylabel=r"$S_2$", title=r"$S_2(h_z)$ (dotted lines = $3\ln2$, $0$)")
    ax[1].set(xlabel="$h_z$", ylabel=r"$dS_2/dh_z$", title="derivative (solid=spline, dotted=FD)")
    ax[0].legend(fontsize=8); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()
else:
    print("no S2 curves loaded -> §2 skipped (extract + pull first; see §1)")

## 3 · Bootstrap the peak position → error on `h_c(L)`

Resample each `S₂` point within its MC error `S2e`, refit the spline, re-locate the peak;
`≥1000` resamples → `σ[h_c(L)]`. Needs finite per-point errors (use `eval_chains≈16` at
extraction). This per-L error feeds the weighted extrapolation in §4.

In [ ]:
rng = np.random.default_rng(0)

def boot_peak(hz, y, ye, B=B_BOOT, s=SMOOTH):
    if not HAVE_SCIPY or not np.all(np.isfinite(ye) & (ye > 0)):
        return np.array([])
    out = np.empty(B)
    for k in range(B):
        yb = y + rng.normal(0, ye)
        hcb, _ = deriv_peak_spline(hz, yb, ye, s=s)
        out[k] = hcb
    return out[np.isfinite(out)]

hc_err = {}
if curves:
    print(f"{'L':>3} {'h_c':>9} {'boot mu':>9} {'sigma':>8} {'16-84%':>20} {'frac_ok':>8}")
    for L in LS_HAVE:
        hz, y, ye = curves[L]
        bc = boot_peak(hz, y, ye)
        if bc.size == 0:
            hc_err[L] = np.nan
            print(f"{L:>3} {hc_spline[L]:>9.4f}   -- no finite-error bootstrap (need finite S2e) --")
            continue
        sd = float(bc.std()); lo, hi = np.percentile(bc, [16, 84])
        hc_err[L] = sd
        print(f"{L:>3} {hc_spline[L]:>9.4f} {bc.mean():>9.4f} {sd:>8.4f} "
              f"[{lo:.4f}, {hi:.4f}] {bc.size/B_BOOT:>8.2f}")
else:
    print("no S2 curves -> §3 skipped")

## 4 · Extrapolation `h_c(L) = h_c + a·L^(−x)` + robustness battery

All-free weighted fit (3 params over L=4,5,6,7 → **1 dof**): report `h_c`, `x`, parameter
covariance, and the fit residual honestly. Then the **robustness battery** — the actual
result is the *spread*, not the single fit: refit with `x` fixed to `{1, 1.59, 2}`, over
all four sizes and dropping L=4. The full range of `h_c` across every variant is the
systematic error.

In [ ]:
def model_free(L, hc, a, x):   return hc + a * np.asarray(L, float) ** (-x)
def model_fixed(x):            return lambda L, hc, a: hc + a * np.asarray(L, float) ** (-x)

hc_summary = {}
if curves and HAVE_SCIPY and len(LS_HAVE) >= 3:
    Lx = np.array(LS_HAVE, float)
    yv = np.array([hc_spline[L] for L in LS_HAVE], float)
    ev = np.array([hc_err[L] for L in LS_HAVE], float)
    have_err = np.all(np.isfinite(ev) & (ev > 0))
    sig = ev if have_err else None
    if not have_err:
        print("[warn] non-finite bootstrap errors -> UNWEIGHTED fits\n")

    # --- all-free fit (needs >=4 pts for a real dof) ---
    print("=== all-free  h_c(L) = h_c + a*L^(-x) ===")
    try:
        p0 = [yv.min(), yv[0] - yv.min(), 1.5]
        popt, pcov = curve_fit(model_free, Lx, yv, p0=p0, sigma=sig, absolute_sigma=have_err, maxfev=20000)
        resid = yv - model_free(Lx, *popt)
        dof = len(Lx) - 3
        chi2 = float(np.sum((resid / ev) ** 2)) if have_err else float("nan")
        perr = np.sqrt(np.diag(pcov))
        print(f"  h_c = {popt[0]:.4f} +- {perr[0]:.4f}")
        print(f"  a   = {popt[1]:.4f} +- {perr[1]:.4f}")
        print(f"  x   = {popt[2]:.4f} +- {perr[2]:.4f}")
        print(f"  dof = {dof}   chi2 = {chi2:.3f}   residuals = {np.round(resid, 4).tolist()}")
        print(f"  cov =\n{np.array2string(pcov, precision=3)}")
        hc_summary["free"] = float(popt[0])
        free_fit = popt
    except Exception as e:
        print("  all-free fit failed:", e); free_fit = None

    # --- robustness battery: x fixed, all-4 and drop-L4 ---
    print("\n=== robustness battery (x fixed) ===")
    print(f"{'variant':>16} {'x':>5} {'h_c':>9} {'h_c_err':>8}")
    battery = []
    subsets = [("all", LS_HAVE)] + ([("drop-L4", [L for L in LS_HAVE if L != 4])]
                                    if 4 in LS_HAVE else [])
    for xname, sub in subsets:
        Ls = np.array(sub, float)
        ys = np.array([hc_spline[L] for L in sub], float)
        es = np.array([hc_err[L] for L in sub], float)
        he = np.all(np.isfinite(es) & (es > 0)); sg = es if he else None
        for x in X_VARIANTS:
            try:
                po, pc = curve_fit(model_fixed(x), Ls, ys, p0=[ys.min(), ys[0]-ys.min()],
                                   sigma=sg, absolute_sigma=he, maxfev=20000)
                pe = np.sqrt(np.diag(pc))[0]
                battery.append(po[0])
                print(f"{xname:>16} {x:>5.2f} {po[0]:>9.4f} {pe:>8.4f}")
            except Exception as e:
                print(f"{xname:>16} {x:>5.2f}   fit failed: {e}")
    if "free" in hc_summary:
        battery.append(hc_summary["free"])
    if battery:
        b = np.array(battery, float)
        hc_summary["spread"] = (float(b.min()), float(b.max()))
        hc_summary["central"] = float(np.median(b))
        print(f"\n==> h_c across ALL variants: [{b.min():.4f}, {b.max():.4f}]  "
              f"(median {np.median(b):.4f}); spread {b.max()-b.min():.4f} = systematic error")
else:
    free_fit = None
    print("§4 skipped: need scipy + >=3 loaded sizes")

## 5 · Comparison with QMC

In [ ]:
print(QMC_NORM_NOTE, "\n")
if hc_summary.get("spread"):
    lo, hi = hc_summary["spread"]; c = hc_summary["central"]
    print(f"S2 pipeline:  h_c = {c:.4f}  in [{lo:.4f}, {hi:.4f}]")
    print(f"QMC:          h_c = {QMC_HC}")
    print(f"offset (central - QMC) = {c - QMC_HC:+.4f}   "
          f"({'consistent' if lo - 0.02 <= QMC_HC <= hi + 0.02 else 'OFFSET beyond battery spread'})")
else:
    print("no S2 extrapolation yet -> comparison skipped")

## 6 · Cross-pipeline figure — S₂ peaks vs FM peaks, both vs `1/L`

Two **independent local estimators** on one axes: `h_c(L)` from S₂ derivative peaks (this
pipeline) and from the FM derivative peaks (`h_c` read straight from the FM aspect-½ JSONs
in `FM_DIR`). Their extrapolations agreeing (or not) with each other and with QMC 0.197 is
the deliverable.

In [ ]:
def load_fm_hc(directory):
    out = {}
    for jp in sorted(glob.glob(os.path.join(directory, "fm_L*.json"))):
        r = json.load(open(jp)); hc = r.get("h_c")
        if hc is not None and np.isfinite(hc):
            out[int(r["L"])] = float(hc)
    return out

fig, ax = plt.subplots(figsize=(7.5, 5))
def _fit_xx(Ldata):   # dense 1/L from the thermodynamic limit out to just past the data
    return np.linspace(0, 1.05 / min(Ldata), 120)

# --- S2 pipeline ---
if curves:
    Ls = np.array(LS_HAVE, float)
    ys = np.array([hc_spline[L] for L in LS_HAVE], float)
    es = np.array([hc_err.get(L, np.nan) for L in LS_HAVE], float)
    ax.errorbar(1/Ls, ys, yerr=np.where(np.isfinite(es), es, 0), fmt="o", ms=7, capsize=3,
                color="C0", label="S$_2$ peak (this work)")
    if free_fit is not None:
        xs = _fit_xx(LS_HAVE)
        ax.plot(xs, free_fit[0] + free_fit[1] * np.where(xs > 0, xs, np.nan) ** free_fit[2],
                "-", color="C0", lw=1.4)
        ax.plot(0, free_fit[0], "*", color="C0", ms=15)
        ax.annotate(f"S$_2$: {free_fit[0]:.3f}", (0, free_fit[0]), fontsize=9,
                    color="C0", xytext=(6, 6), textcoords="offset points")

# --- FM pipeline ---
fm_hc = load_fm_hc(FM_DIR)
if fm_hc:
    Lf = np.array(sorted(fm_hc), float); yf = np.array([fm_hc[int(L)] for L in Lf], float)
    ax.plot(1/Lf, yf, "s", ms=7, color="C3", label="FM peak (aspect-½)")
    if HAVE_SCIPY and len(Lf) >= 3:
        try:
            pf, _ = curve_fit(model_free, Lf, yf, p0=[yf.min(), yf[0]-yf.min(), 1.5], maxfev=20000)
            xf = _fit_xx(sorted(fm_hc))
            ax.plot(xf, pf[0] + pf[1] * np.where(xf > 0, xf, np.nan) ** pf[2], "-", color="C3", lw=1.4)
            ax.plot(0, pf[0], "*", color="C3", ms=15)
            ax.annotate(f"FM: {pf[0]:.3f}", (0, pf[0]), fontsize=9, color="C3",
                        xytext=(6, -12), textcoords="offset points")
        except Exception as e:
            print("FM fit failed:", e)
else:
    print(f"(no FM curves in {FM_DIR} -> FM overlay skipped)")

ax.axhline(QMC_HC, ls="--", color="green", lw=1.4, label=f"QMC={QMC_HC}")
ax.set(xlabel=r"$1/L$", ylabel=r"$h_z^c(L)$",
       title="Two independent local locators of $h_z^c$ (hx=0)  —  S$_2$ vs FM")
ax.set_xlim(-0.012, 0.27); ax.legend(fontsize=9); plt.tight_layout(); plt.show()